In [ ]:
# Lab type: review
# Course: AI401 — AI Applications with LLMs
# Lesson: Orchestration Patterns: When to Use Frameworks and When Not To
# Task: Compare a LangChain implementation and a direct-API async implementation of the same batch classification task, then answer judgment questions

# Lab: Reviewing Orchestration Approaches

Two engineers have implemented the same batch ticket classification pipeline. Implementation A uses LangChain; Implementation B uses the Anthropic SDK directly with async/await and bounded concurrency.

Both produce correct classifications. Your task: read both implementations, run the inspection cells, and answer the judgment questions.

## Setup

In [ ]:
import asyncio
import time
import anthropic

# Sample tickets for testing
TICKETS = [
    "Payment system down — all transactions failing",
    "Package not delivered, tracking shows it's still in transit",
    "App crashes immediately on login since yesterday's update",
    "Invoice shows incorrect amount, I was charged twice",
    "How do I update my billing address?",
    "Subscription auto-renewed but I cancelled last month",
    "Dashboard not loading for the last two hours",
    "Delivery estimated 3 days ago — nothing arrived",
]

CLASSIFY_SYSTEM = (
    "Classify the support ticket into exactly one of: billing, technical, shipping, other. "
    "Think through the ticket content, then state your classification "
    "on a new line in this exact format: CLASSIFICATION: <label>"
)

## Implementation A: LangChain

In [ ]:
# Implementation A uses LangChain's ChatAnthropic + batch processing
# NOTE: This cell shows the structure — it will raise ImportError if
# langchain_anthropic is not installed, which is expected in this environment.
# You are reviewing the code, not running it.

IMPL_A_CODE = '''
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

def classify_batch_langchain(tickets: list[str]) -> list[str]:
    llm = ChatAnthropic(
        model="claude-haiku-4-5-20251001",
        max_tokens=128,
    )
    messages_batch = [
        [SystemMessage(content=CLASSIFY_SYSTEM), HumanMessage(content=t)]
        for t in tickets
    ]
    responses = llm.batch(messages_batch)
    results = []
    for r in responses:
        text = r.content
        import re
        match = re.search(r"CLASSIFICATION: (\\w+)", text)
        results.append(match.group(1).lower() if match else "unknown")
    return results
'''

print('Implementation A — LangChain batch:')
print(IMPL_A_CODE)

## Implementation B: Direct Anthropic SDK with async

In [ ]:
import re

async_client = anthropic.AsyncAnthropic()


async def classify_ticket(ticket: str, semaphore: asyncio.Semaphore) -> dict:
    """Classify a single ticket. Bounded by semaphore to avoid rate-limit bursts."""
    async with semaphore:
        t0 = time.perf_counter()
        response = await async_client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=128,
            system=CLASSIFY_SYSTEM,
            messages=[{'role': 'user', 'content': ticket}],
        )
        latency_ms = round((time.perf_counter() - t0) * 1000, 1)
        text = response.content[0].text
        match = re.search(r'CLASSIFICATION: (\w+)', text)
        return {
            'ticket': ticket[:50],
            'category': match.group(1).lower() if match else 'unknown',
            'input_tokens': response.usage.input_tokens,
            'output_tokens': response.usage.output_tokens,
            'latency_ms': latency_ms,
        }


async def classify_batch_direct(tickets: list[str], max_concurrent: int = 5) -> list[dict]:
    """Classify all tickets with bounded concurrency."""
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [classify_ticket(t, semaphore) for t in tickets]
    return await asyncio.gather(*tasks)

## Inspection: run Implementation B

In [ ]:
t0 = time.perf_counter()
results = asyncio.run(classify_batch_direct(TICKETS, max_concurrent=5))
wall_time = time.perf_counter() - t0

print(f'Classified {len(results)} tickets in {wall_time:.2f}s')
print()
for r in results:
    print(f"{r['category']:12s}  {r['latency_ms']:6.0f}ms  "
          f"in={r['input_tokens']:3d} out={r['output_tokens']:3d}  "
          f"{r['ticket']}")

In [ ]:
# Compute per-call statistics
latencies = [r['latency_ms'] for r in results]
input_tokens = [r['input_tokens'] for r in results]
output_tokens = [r['output_tokens'] for r in results]

print(f'Latency  — p50: {sorted(latencies)[len(latencies)//2]:.0f}ms, '
      f'max: {max(latencies):.0f}ms')
print(f'Input tokens   — avg: {sum(input_tokens)/len(input_tokens):.0f}')
print(f'Output tokens  — avg: {sum(output_tokens)/len(output_tokens):.0f}')

## Judgment question 1

> Implementation A hides the token count per request inside LangChain's internals. At what processing scale does this opacity become a practical problem, and what specific information are you missing that would help you control costs?

In [ ]:
# Your answer:
#
# Scale threshold:
#
# Missing information:
#

## Judgment question 2

> The semaphore in `classify_batch_direct` is set to `max_concurrent=5`. The comments in Lesson 7 suggest: rate_limit_rpm / (60 / avg_latency_seconds).

> Using the latency data from your run above, what `max_concurrent` would be appropriate for a 60 RPM rate limit tier?

In [ ]:
# Calculate the appropriate max_concurrent for a 60 RPM limit
avg_latency_s = sum(latencies) / len(latencies) / 1000
rpm_limit = 60

# Your calculation:
# max_concurrent = rpm_limit * avg_latency_s / 60
max_concurrent_60rpm = rpm_limit * avg_latency_s / 60
print(f'Average latency : {avg_latency_s:.3f}s')
print(f'Max concurrent  : {max_concurrent_60rpm:.2f} → use {max(1, int(max_concurrent_60rpm))}')

## Judgment question 3

> Implementation B logs `latency_ms` per call. What 3 additional fields would you add to the per-call result dict to make it possible to diagnose a performance regression in production?

In [ ]:
# Your answer — list 3 fields with types and the diagnostic question each answers:
#
# Field 1:
#
# Field 2:
#
# Field 3:
#

## Judgment question 4

> The lesson's framework adoption criteria are:
> - The pattern is well-understood and well-tested in the framework
> - You don't need visibility into the details the framework hides
> - The team has read and understands the framework source

> Name one scenario in this classification pipeline where you would choose Implementation A (LangChain) over Implementation B, and justify it against the three criteria.

In [ ]:
# Your answer:
#
# Scenario:
#
# Justification against each criterion:
#  1.
#  2.
#  3.
#